# Week 3 Day 3: Demystifying Tokenizers 🔢

**What are Tokenizers?**
Large Language Models do not read text like humans do. They only understand numbers. A **Tokenizer** acts as a dictionary that breaks text down into small chunks (tokens) and converts them into unique ID numbers.

**Key Concepts Today:**
1. **Encoding & Decoding:** Turning text into numbers, and numbers back into text.
2. **Vocab Size:** How many unique tokens a model knows.
3. **Chat Templates:** How "Instruct" models format conversations.
4. **Different Models = Different Tokenizers:** We will compare Llama 3.1, Phi-3, Qwen2, and StarCoder2.

In [1]:
# Install transformers (the library that holds AutoTokenizer)
!pip install -q transformers

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer

# 1. Load the HuggingFace token from your .env file
load_dotenv(find_dotenv(), override=True)
hf_token = os.environ.get("HF_TOKEN")

# 2. Login to HuggingFace
# IMPORTANT: Llama 3.1 requires you to agree to Meta's license on HuggingFace first!
# Go to: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B and click "Agree"
login(hf_token, add_to_git_credential=True)
print("Logged in successfully!")

`add_to_git_credential=True` is only supported when a token is passed directly. It is ignored by the browser-based login.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Logged in successfully!


## 1. Encoding and Decoding with Llama 3.1
Let's load the tokenizer for Meta's Llama 3.1 model. 
*Note: We are ONLY loading the tokenizer, not the massive LLM itself. This runs instantly on CPU!*

In [3]:
# Load the Llama 3.1 Tokenizer
# trust_remote_code=True is sometimes required for newer models to execute custom tokenizer code
llama_tokenizer = AutoTokenizer.from_pretrained(
    'meta-llama/Meta-Llama-3.1-8B', 
    trust_remote_code=True
)
print("Llama 3.1 Tokenizer loaded!")

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Llama 3.1 Tokenizer loaded!


In [4]:
# The text we want to tokenize
text = "I am excited to show Tokenizers in action to my LLM engineers"

# .encode() converts the string into a list of integer Token IDs
tokens = llama_tokenizer.encode(text)

print("Original Text:", text)
print("Token IDs:    ", tokens)
print("Total Tokens: ", len(tokens))

Original Text: I am excited to show Tokenizers in action to my LLM engineers
Token IDs:     [128000, 40, 1097, 12304, 311, 1501, 9857, 12509, 304, 1957, 311, 856, 445, 11237, 25175]
Total Tokens:  15


In [8]:
# .decode() turns the integer IDs back into a single string
decoded_text = llama_tokenizer.decode(tokens)
print("Decoded string:\n", decoded_text)

print("-" * 50)

# .batch_decode() shows us EXACTLY how the text was chopped up!
# It decodes each token individually and puts them in a list.
# Notice how some words are kept whole (e.g. ' excited'), while others are split!
chopped_text = llama_tokenizer.batch_decode(tokens)
print("How the tokenizer chopped it up:\n", chopped_text)

Decoded string:
 <|begin_of_text|>I am excited to show Tokenizers in action to my LLM engineers
--------------------------------------------------
How the tokenizer chopped it up:
 ['<|begin_of_text|>I am excited to show Tokenizers in action to my LLM engineers']


In [9]:
# To see EXACTLY how the tokenizer chopped the text, we must decode each ID individually
print("Token ID  |  Decoded Text")
print("-" * 30)
for token_id in tokens:
    # decode() converts a single ID back to text
    text_chunk = llama_tokenizer.decode([token_id])
    print(f"{token_id:<9} | '{text_chunk}'")

Token ID  |  Decoded Text
------------------------------
128000    | '<|begin_of_text|>'
40        | 'I'
1097      | ' am'
12304     | ' excited'
311       | ' to'
1501      | ' show'
9857      | ' Token'
12509     | 'izers'
304       | ' in'
1957      | ' action'
311       | ' to'
856       | ' my'
445       | ' L'
11237     | 'LM'
25175     | ' engineers'


## 2. Chat Templates (Instruct Models)

Many models have an **"Instruct"** variant (e.g., `Meta-Llama-3.1-8B-Instruct`).
These models have been specifically trained for chat interfaces. They expect your prompt to be formatted in a very strict, specific way with special invisible tokens denoting the `system`, `user`, and `assistant` roles.

Instead of typing out these complex formats manually, HuggingFace tokenizers have a built-in method called **`apply_chat_template()`** that formats our standard message list perfectly for whichever model we are using!

In [10]:
# Notice the "-Instruct" at the end!
llama_instruct_tokenizer = AutoTokenizer.from_pretrained(
    'meta-llama/Meta-Llama-3.1-8B-Instruct', 
    trust_remote_code=True
)
print("Llama 3.1 Instruct Tokenizer loaded!")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Llama 3.1 Instruct Tokenizer loaded!


In [11]:
# The standard OpenAI-style message format we used in Week 1 & 2
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
]

# Let the tokenizer convert our messages into Llama's strict prompt format!
# tokenize=False → Return the raw string (not numbers) so we can read it.
# add_generation_prompt=True → Adds the final "Assistant:" tag at the end so the model knows it's time to reply.
prompt = llama_instruct_tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

print("How Llama 3.1 ACTUALLY sees your chat:")
print("-" * 50)
print(prompt)

How Llama 3.1 ACTUALLY sees your chat:
--------------------------------------------------
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell a light-hearted joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>




## 3. Comparing Tokenizers Across Models

Not all LLMs use the same dictionary! 
The number `100` might mean "apple" to Llama, but it might mean "spaceship" to an Alibaba model.
Let's load three other popular models and see how their tokenizers chop up the exact same text.

**The Models:**
1. **Phi-3** — Microsoft's highly efficient small model.
2. **Qwen2** — Alibaba Cloud's incredibly powerful open-source model.
3. **StarCoder2** — A model trained specifically on programming code (by ServiceNow, HF, & Nvidia).

In [12]:
# Define the HuggingFace paths for our three competitor models
PHI3_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
QWEN2_MODEL_NAME = "Qwen/Qwen2-7B-Instruct"
STARCODER2_MODEL_NAME = "bigcode/starcoder2-3b"

# Load their tokenizers
phi3_tokenizer = AutoTokenizer.from_pretrained(PHI3_MODEL_NAME)
qwen2_tokenizer = AutoTokenizer.from_pretrained(QWEN2_MODEL_NAME)
# StarCoder2 requires trust_remote_code=True
starcoder2_tokenizer = AutoTokenizer.from_pretrained(STARCODER2_MODEL_NAME, trust_remote_code=True)

print("All competitor tokenizers loaded!")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

All competitor tokenizers loaded!


In [13]:
# Let's see how each model translates the exact same sentence into numbers
text = "I am excited to show Tokenizers in action to my LLM engineers"

print("Llama 3.1: ", llama_tokenizer.encode(text))
print("Phi-3:     ", phi3_tokenizer.encode(text))
print("Qwen2:     ", qwen2_tokenizer.encode(text))

Llama 3.1:  [128000, 40, 1097, 12304, 311, 1501, 9857, 12509, 304, 1957, 311, 856, 445, 11237, 25175]
Phi-3:      [306, 626, 24173, 304, 1510, 25159, 19427, 297, 3158, 304, 590, 365, 26369, 6012, 414]
Qwen2:      [40, 1079, 12035, 311, 1473, 9660, 12230, 304, 1917, 311, 847, 444, 10994, 24198]


In [14]:
# How does each model format the exact same Chat Messages?
print("--- Llama 3.1 Chat Template ---")
print(llama_instruct_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\n")

print("--- Phi-3 Chat Template ---")
print(phi3_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\n")

print("--- Qwen2 Chat Template ---")
print(qwen2_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

--- Llama 3.1 Chat Template ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell a light-hearted joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>




--- Phi-3 Chat Template ---
<|system|>
You are a helpful assistant<|end|>
<|user|>
Tell a light-hearted joke for a room of Data Scientists<|end|>
<|assistant|>



--- Qwen2 Chat Template ---
<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Tell a light-hearted joke for a room of Data Scientists<|im_end|>
<|im_start|>assistant



In [15]:
# StarCoder2 was trained specifically on code. 
# Let's see how its tokenizer handles Python indentation!
code = """
def hello_world(person):
  print("Hello", person)
"""

tokens = starcoder2_tokenizer.encode(code)
print("StarCoder2 Token Breakdown:")
print("-" * 30)
for token in tokens:
    # We use repr() to clearly see spaces, tabs, and newlines
    text_chunk = starcoder2_tokenizer.decode([token])
    print(f"{token:<9} | {repr(text_chunk)}")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


StarCoder2 Token Breakdown:
------------------------------
222       | '\n'
610       | 'def'
17966     | ' hello'
100       | '_'
5879      | 'world'
45        | '('
6427      | 'person'
731       | '):'
353       | '\n '
1489      | ' print'
459       | '("'
8302      | 'Hello'
411       | '",'
4944      | ' person'
46        | ')'
222       | '\n'
